In [1]:
import pandas as pd
import os
import xarray as xr

## Data Reading

In [14]:
df_mapping = pd.read_csv("../../data/super_processed/4_no2_to_traffic_sensor_mapping.csv")
df_air = pd.read_csv("../../data/super_processed/6_df_air_data_and_locations_reduced.csv")

Read traffic data

In [3]:
# Define the root directory for processed traffic data
root_dir = '../../data/super_processed/5_traffic'

# List to hold DataFrames
dataframes = []

# Iterate through all folders in the root directory
for folder_name in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, folder_name)
    
    # Check if the folder exists and is a directory
    if os.path.isdir(folder_path):
        print(f"Processing folder: {folder_path}")
        
        # Iterate through all files in the folder
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.parquet'):  # Ensure it's a Parquet file
                file_path = os.path.join(folder_path, file_name)
                
                # Read the Parquet file
                try:
                    df = pd.read_parquet(file_path)
                    print(f"Successfully read: {file_path} with {len(df)} rows.")
                    
                    # Append the DataFrame to the list
                    dataframes.append(df)
                    
                except Exception as e:
                    print(f"Error reading file {file_name}: {e}")

# Concatenate all DataFrames into one
if dataframes:  # Check if the list is not empty
    df_traffic = pd.concat(dataframes, ignore_index=True)
    print(f"Combined DataFrame created with {len(df_traffic)} rows.")

Processing folder: ../../data/super_processed/5_traffic/2022
Successfully read: ../../data/super_processed/5_traffic/2022/12-2022_processed.parquet with 48259 rows.
Successfully read: ../../data/super_processed/5_traffic/2022/11-2022_processed.parquet with 46690 rows.
Successfully read: ../../data/super_processed/5_traffic/2022/10-2022_processed.parquet with 47556 rows.
Successfully read: ../../data/super_processed/5_traffic/2022/01-2022_processed.parquet with 44145 rows.
Successfully read: ../../data/super_processed/5_traffic/2022/06-2022_processed.parquet with 42196 rows.
Successfully read: ../../data/super_processed/5_traffic/2022/05-2022_processed.parquet with 42735 rows.
Successfully read: ../../data/super_processed/5_traffic/2022/08-2022_processed.parquet with 45087 rows.
Successfully read: ../../data/super_processed/5_traffic/2022/02-2022_processed.parquet with 40278 rows.
Successfully read: ../../data/super_processed/5_traffic/2022/03-2022_processed.parquet with 44372 rows.
Suc

In [4]:
# Define the root directory for processed traffic data
root_dir = '../../data/raw/meteo'

# List to hold DataFrames
dataframes = []

# Iterate through all folders in the root directory
for folder_name in os.listdir(root_dir):
    folder_path = os.path.join(root_dir, folder_name)

    if folder_name == '2013' or folder_name == '2014' or folder_name == '2015' or folder_name == '2016' or folder_name == '2017':
        continue
    
    # Check if the folder exists and is a directory
    if os.path.isdir(folder_path):
        print(f"Processing folder: {folder_path}")
        
        # Iterate through all files in the folder
        for file_name in os.listdir(folder_path):
            if file_name.endswith('.grib'):  # Ensure it's a GRIB file
                file_path = os.path.join(folder_path, file_name)
                
                # Read the file
                try:                    
                    df = xr.open_dataset(file_path, engine='cfgrib',  backend_kwargs={'indexpath': None})
                    print(f"Successfully read: {file_path} with {len(df)} rows.")
                    
                    df = df.to_dataframe().reset_index()  # Reset index if needed
                    df = df[df['d2m'].notna()]
                    
                    # Append the DataFrame to the list
                    dataframes.append(df)
                    
                except Exception as e:
                    print(f"Error reading file {file_name}: {e}")

# Concatenate all DataFrames into one
if dataframes:  # Check if the list is not empty
    df_meteo = pd.concat(dataframes, ignore_index=True)
    print(f"Combined DataFrame created with {len(df_meteo)} rows.")

Processing folder: ../../data/raw/meteo/2022
Successfully read: ../../data/raw/meteo/2022/4e143763b6ddd90830b1b2e53ae3d7a5.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/84bbb1de389f549dc6756798501cdca5.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/93003d3f27f9fd461aad3ee6e8f7bce0.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/e3aa8ebebab5e5eca9010c191fca712.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/41df651812e856531f6592e080a66fce.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/d9331ddced57882f0d634e4db7abc227.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/b3fcc9fe6bc5bd11659b6ba730d19df4.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/773f78fa31108dc1bc9242648ce689bd.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/b81359a8caf89dcca96e37ff9c93a920.grib with 8 rows.
Successfully read: ../../data/raw/meteo/2022/17d2a65169a31294a6845abb2e3382f6.grib with 8 ro

## Data Cleaning

In [15]:
df_air['id_no2'] = df_air['id_no2'].astype(str)
df_mapping['id_trafico'] = df_mapping['id_trafico'].astype(str)
df_mapping['id_no2'] = df_mapping['id_no2'].astype(str)

In [16]:
df_traffic = df_traffic.rename(columns = {'hora': 'fecha'})

save traffic_data

In [67]:
df_traffic.to_parquet("7_0_all_traffic.parquet", index = False)

In [ ]:
df_meteo = df_meteo.rename(columns = {'valid_time':'fecha'})
df_meteo = df_meteo.drop(columns = ['time','step','surface','number'])

save meteo data

In [22]:
df_meteo.to_parquet("7_1_all_meteo.parquet", index = False)

vamos a quedarnos solo con las estaciones que nos interesan (de momento):

estacion 1

- latitud: 40.5
- longitud -3.7

TODO: posteriormente podemos sacar datos mas precisos.

In [23]:
df_meteo = df_meteo[(df_meteo['latitude'] == 40.5) & (df_meteo['longitude'] == -3.7)]

get the unique values of the longitude and latitude of the air quality data


In [25]:
df_air_locations = df_air[['longitud','latitud']].drop_duplicates().reset_index(drop = True)
df_meteo_locations = df_meteo[['longitude','latitude']].drop_duplicates().reset_index(drop = True)

In [26]:
import folium

# Crear un mapa centrado en la ubicación promedio
center_lat = (df_air_locations['latitud'].mean() + df_meteo_locations['latitude'].mean()) / 2
center_lon = (df_air_locations['longitud'].mean() + df_meteo_locations['longitude'].mean()) / 2

# Crear el mapa base
m = folium.Map(location=[center_lat, center_lon], zoom_start=11, 
               tiles='CartoDB positron')

# Añadir marcadores para todas las estaciones de calidad del aire (azul)
for idx, row in df_air_locations.iterrows():
    folium.Marker(
        location=[row['latitud'], row['longitud']],
        popup=f"""
        <b>Estación de aire #{idx}</b><br>
        Latitud: {row['latitud']:.6f}<br>
        Longitud: {row['longitud']:.6f}
        """,
        icon=folium.Icon(color='blue', icon='info-sign'),
        tooltip="Estación de calidad del aire"
    ).add_to(m)

# Añadir marcadores para todas las estaciones meteorológicas (rojo)
for idx, row in df_meteo_locations.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"""
        <b>Estación meteorológica #{idx}</b><br>
        Latitud: {row['latitude']:.6f}<br>
        Longitud: {row['longitude']:.6f}
        """,
        icon=folium.Icon(color='red', icon='cloud'),
        tooltip="Estación meteorológica"
    ).add_to(m)

# Añadir leyenda al mapa
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; right: 50px; width: 220px; height: 90px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; padding: 10px;
     border-radius: 5px;">
     &nbsp; <b>Leyenda</b> <br>
     &nbsp; <i class="fa fa-info-sign fa-2x" style="color:blue"></i>&nbsp; Estaciones de aire<br>
     &nbsp; <i class="fa fa-cloud fa-2x" style="color:red"></i>&nbsp; Estaciones meteorológicas
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

# Mostrar el mapa
m

In [27]:
df_meteo = df_meteo.rename(columns = {'longitude':'longitud_meteo','latitude':'latitud_meteo'})

In [28]:
df_meteo

,latitud_meteo,longitud_meteo,fecha,d2m,t2m,ssr,ssrd,u10,v10,sp,tp
4,40.5,-3.7,2022-09-01 00:00:00,280.542236,292.866821,20434510.0,24966316.0,1.254089,0.034668,93391.2500,4.410743e-07
14,40.5,-3.7,2022-09-01 01:00:00,280.599976,292.535706,0.0,0.0,0.903183,-0.534027,93461.1875,0.000000e+00
24,40.5,-3.7,2022-09-01 02:00:00,280.711426,291.569275,0.0,0.0,0.506042,-0.807327,93441.0000,0.000000e+00
34,40.5,-3.7,2022-09-01 03:00:00,280.734070,290.779358,0.0,0.0,0.439850,-0.815460,93424.0625,0.000000e+00
44,40.5,-3.7,2022-09-01 04:00:00,280.815918,289.865906,0.0,0.0,0.460663,-0.967896,93422.3750,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...
601874,40.5,-3.7,2018-03-30 19:00:00,271.953674,278.594971,15735110.0,18898142.0,3.343079,3.779297,91692.7500,2.062368e-03
601884,40.5,-3.7,2018-03-30 20:00:00,273.899292,276.888916,15735110.0,18898142.0,3.634674,3.782806,91706.6875,3.013182e-03
601894,40.5,-3.7,2018-03-30 21:00:00,274.979126,275.986572,15735110.0,18898142.0,3.533691,3.429565,91745.3125,3.689832e-03
601904,40.5,-3.7,2018-03-30 22:00:00,275.429993,276.300293,15735110.0,18898142.0,3.531952,2.684830,91769.3125,3.814471e-03


In [29]:
df_meteo.to_parquet("7_2_meteo_data_one_station.parquet", index = False)

Joining the data `air_quality`, `traffic` and `meteo`

In [37]:
df_air.rename(columns = {'longitud':'longitud_no2', 'latitud':'latitud_no2'}, inplace = True)

In [81]:
df = pd.merge(df_air, df_mapping, how = 'left', left_on='id_no2', right_on='id_no2')

In [82]:
df['fecha'] = pd.to_datetime(df['fecha'])
df_traffic['fecha'] = pd.to_datetime(df_traffic['fecha'])
df_meteo['fecha'] = pd.to_datetime(df_meteo['fecha'])

In [83]:
df = pd.merge(df, df_traffic, how = 'inner', left_on=['id_trafico','fecha'], right_on=['id_trafico','fecha'])

In [84]:
df = pd.merge(df, df_meteo, how = 'inner', left_on=['fecha'], right_on=['fecha'])

In [50]:
df.to_parquet("7_3_no2_with_traffic_and_meteo.parquet", index=False)

## Gestionar outliers y valores faltantes

In [51]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# from matplotlib.colors import LinearSegmentedColormap

# # Filtrar para el ID específico
# df_sensor = df[df['id_trafico'] == '5084'].copy()

# # Asegurarse que las fechas estén en formato datetime
# fecha_col = 'fecha'  # Ajusta al nombre real de tu columna de fecha
# if not pd.api.types.is_datetime64_any_dtype(df_sensor[fecha_col]):
#     df_sensor[fecha_col] = pd.to_datetime(df_sensor[fecha_col])

# # Crear un índice de todas las horas que deberían existir
# fecha_min = df_sensor[fecha_col].min()
# fecha_max = df_sensor[fecha_col].max()
# todas_horas = pd.date_range(start=fecha_min, end=fecha_max, freq='H')

# # Crear DataFrame con todas las horas
# df_completo = pd.DataFrame(index=todas_horas)
# df_completo.index.name = 'hora'

# # Marcar las horas que existen en los datos originales
# df_sensor_hora = df_sensor.set_index(fecha_col)
# df_completo['tiene_datos'] = df_completo.index.isin(df_sensor_hora.index).astype(int)

# # Calcular estadísticas de completitud
# total_horas = len(todas_horas)
# horas_con_datos = df_completo['tiene_datos'].sum()
# porcentaje_completitud = (horas_con_datos / total_horas) * 100

# print(f"Periodo: {fecha_min} a {fecha_max}")
# print(f"Total de horas en el periodo: {total_horas}")
# print(f"Horas con datos: {horas_con_datos} ({porcentaje_completitud:.2f}%)")
# print(f"Horas sin datos: {total_horas - horas_con_datos} ({100-porcentaje_completitud:.2f}%)")

# # Visualizar huecos temporales por día y hora
# df_completo['fecha'] = df_completo.index.date
# df_completo['hora_dia'] = df_completo.index.hour

# # Crear matriz para heatmap (días x horas)
# matriz_datos = df_completo.pivot_table(
#     index=df_completo.index.date, 
#     columns='hora_dia', 
#     values='tiene_datos', 
#     aggfunc='first'
# )

# # Crear mapa de calor para visualizar presencia/ausencia de datos
# plt.figure(figsize=(16, 10))
# cmap = LinearSegmentedColormap.from_list('datos', ['#FFFFFF', '#4CAF50'])
# ax = sns.heatmap(matriz_datos, cmap=cmap, cbar_kws={'label': 'Datos disponibles'})
# ax.set_title(f'Disponibilidad de datos por hora - Sensor {10885}', fontsize=14)
# ax.set_xlabel('Hora del día', fontsize=12)
# ax.set_ylabel('Fecha', fontsize=12)
# plt.tight_layout()
# plt.show()

In [52]:
# import calplot
# import matplotlib.pyplot as plt

# # Preparar datos para el calendario
# df_completo['fecha'] = pd.to_datetime(df_completo['fecha'])
# datos_diarios = df_completo.groupby('fecha')['tiene_datos'].sum()
# datos_diarios = datos_diarios / 24 * 100  # Convertir a porcentaje de completitud

# # Crear visualización de calendario
# plt.figure(figsize=(16, 10))
# calplot.calplot(datos_diarios, cmap='YlGn', 
#                fillcolor='whitesmoke',
#                vmin=0, vmax=100, 
#                suptitle=f'Disponibilidad diaria de datos (%) - Sensor {10885}')
# plt.tight_layout()
# plt.show()

In [85]:
# Ver cuántos id_trafico están asignados a cada id_no2
resumen = df.groupby('id_no2')['id_trafico'].nunique().reset_index()
resumen.columns = ['id_no2', 'num_sensores_trafico']
print("Número de sensores de tráfico asignados a cada sensor NO2:")
print(resumen.sort_values(by='num_sensores_trafico', ascending=False))

# Identificar sensores NO2 sin sensores de tráfico asignados
sin_trafico = resumen[resumen['num_sensores_trafico'] == 0]['id_no2'].tolist()
if sin_trafico:
    print(f"\nSensores NO2 sin sensores de tráfico asignados: {sin_trafico}")

Número de sensores de tráfico asignados a cada sensor NO2:
      id_no2  num_sensores_trafico
12  28079056                    14
0   28079004                    10
2   28079011                     8
7   28079039                     7
5   28079036                     6
1   28079008                     5
4   28079035                     4
10  28079048                     4
6   28079038                     3
3   28079016                     2
8   28079040                     2
9   28079047                     2
11  28079050                     2


In [86]:
# Para cada id_no2, quedarse solo con el id_trafico que tenga mas datos y añadir una nueva columna con el numero de datos, el inicio y el fin de los dato

# Paso 1: Calcular métricas para cada combinación id_no2 y id_trafico
stats_trafico = df.groupby(['id_no2', 'id_trafico']).agg(
    num_registros=('fecha', 'count'),
    fecha_inicio=('fecha', 'min'),
    fecha_fin=('fecha', 'max')
).reset_index()

# Paso 2: Para cada id_no2, encontrar el id_trafico con más datos
mejor_trafico = stats_trafico.sort_values(['id_no2', 'num_registros'], ascending=[True, False])
mejor_trafico = mejor_trafico.groupby('id_no2').first().reset_index()

# Paso 3: Añadir columnas informativas
mejor_trafico['periodo_dias'] = (mejor_trafico['fecha_fin'] - mejor_trafico['fecha_inicio']).dt.days
mejor_trafico['densidad_datos'] = mejor_trafico['num_registros'] / mejor_trafico['periodo_dias'].clip(lower=1)

# Paso 4: Unir con el DataFrame original para mantener solo las filas con mejores sensores
df_filtrado = pd.merge(
    df,
    mejor_trafico[['id_no2', 'id_trafico']],
    on=['id_no2', 'id_trafico'],
    how='inner'
)

# Paso 5: Añadir las métricas calculadas al DataFrame filtrado
df_final = pd.merge(
    df_filtrado,
    mejor_trafico[['id_no2', 'id_trafico', 'num_registros', 'fecha_inicio', 'fecha_fin', 'periodo_dias', 'densidad_datos']],
    on=['id_no2', 'id_trafico'],
    how='left'
)

# Mostrar resumen del resultado
print(f"DataFrame original: {len(df)} filas, {df['id_no2'].nunique()} sensores NO2, {df['id_trafico'].nunique()} sensores tráfico")
print(f"DataFrame filtrado: {len(df_final)} filas, {df_final['id_no2'].nunique()} sensores NO2, {df_final['id_trafico'].nunique()} sensores tráfico")

# Mostrar información sobre los mejores sensores de tráfico seleccionados
print("\nResumen de mejores sensores de tráfico seleccionados:")
resumen = mejor_trafico[['id_no2', 'id_trafico', 'num_registros', 'periodo_dias']].sort_values('num_registros', ascending=False)
print(resumen.head(20))  # Muestra los 10 primeros

# df_final.to_csv('datos_filtrados_mejor_trafico.csv', index=False)

DataFrame original: 3228482 filas, 13 sensores NO2, 69 sensores tráfico
DataFrame filtrado: 736545 filas, 13 sensores NO2, 13 sensores tráfico

Resumen de mejores sensores de tráfico seleccionados:
      id_no2 id_trafico  num_registros  periodo_dias
10  28079048       4461          59056          2525
11  28079050       5465          58952          2525
8   28079040       5783          58926          2525
9   28079047       4129          58486          2525
6   28079038       4472          58168          2525
7   28079039       5422          58165          2525
12  28079056       5084          58112          2525
1   28079008       4022          57979          2525
2   28079011       3911          57580          2525
5   28079036       6116          56793          2525
3   28079016       3791          55989          2525
4   28079035       3731          50675          2525
0   28079004       4284          47664          2525


In [87]:
# filtrar el df original para quedarnos con estos id_trafico y los id_no2 que tengan asignados
df_filtered = df[df['id_trafico'].isin(mejor_trafico['id_trafico'])]
df_filtered = df_filtered[df_filtered['id_no2'].isin(mejor_trafico['id_no2'])]

In [26]:
#df_filtered.to_parquet("no2_with_traffic_and_meteo_one_station_filtered_with_best_trafic_id.parquet", index=False)

In [65]:
df_filtered.to_parquet("7_4_no2_with_traffic_and_1meteo_and_1trafic_id.parquet", index=False)

In [88]:
# Drop all columns from df_meteo except 'fecha'
cols_to_drop = [col for col in df_meteo.columns if col != 'fecha']

In [91]:
df_filtered = df_filtered.drop(columns=cols_to_drop)
df_filtered.to_parquet("7_5_no2_with_1traffic_id.parquet", index=False)

In [77]:
df_filtered

,id_no2,fecha,no2_value,longitud_no2,latitud_no2,id_trafico,distance_m,intensidad,carga,ocupacion,vmed
0,28079004,2018-01-01 01:00:00,15.0,-3.712257,40.423882,4284,56.2,480.75,14.734789,3.805512,0.0
7,28079004,2018-01-01 02:00:00,35.0,-3.712257,40.423882,4284,56.2,613.25,20.041989,5.289849,0.0
14,28079004,2018-01-01 03:00:00,29.0,-3.712257,40.423882,4284,56.2,505.75,16.372219,4.331191,0.0
21,28079004,2018-01-01 04:00:00,16.0,-3.712257,40.423882,4284,56.2,408.50,13.328641,3.337209,0.0
28,28079004,2018-01-01 05:00:00,12.0,-3.712257,40.423882,4284,56.2,312.00,10.043269,1.713942,0.0
...,...,...,...,...,...,...,...,...,...,...,...
3291225,28079056,2024-11-30 20:00:00,99.0,-3.718768,40.385034,5084,170.1,1121.75,39.433697,7.186539,0.0
3291235,28079056,2024-11-30 21:00:00,111.0,-3.718768,40.385034,5084,170.1,1056.25,36.892781,6.032426,0.0
3291245,28079056,2024-11-30 22:00:00,69.0,-3.718768,40.385034,5084,170.1,877.25,30.381305,4.107438,0.0
3291255,28079056,2024-11-30 23:00:00,54.0,-3.718768,40.385034,5084,170.1,665.25,22.826005,2.515596,0.0


In [66]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import numpy as np
# from matplotlib.colors import LinearSegmentedColormap

# # Filtrar para el ID específico
# df_sensor = df[df['id_trafico'] == '5465'].copy()

# # Asegurarse que las fechas estén en formato datetime
# fecha_col = 'fecha'  # Ajusta al nombre real de tu columna de fecha
# if not pd.api.types.is_datetime64_any_dtype(df_sensor[fecha_col]):
#     df_sensor[fecha_col] = pd.to_datetime(df_sensor[fecha_col])

# # Crear un índice de todas las horas que deberían existir
# fecha_min = df_sensor[fecha_col].min()
# fecha_max = df_sensor[fecha_col].max()
# todas_horas = pd.date_range(start=fecha_min, end=fecha_max, freq='H')

# # Crear DataFrame con todas las horas
# df_completo = pd.DataFrame(index=todas_horas)
# df_completo.index.name = 'hora'

# # Marcar las horas que existen en los datos originales
# df_sensor_hora = df_sensor.set_index(fecha_col)
# df_completo['tiene_datos'] = df_completo.index.isin(df_sensor_hora.index).astype(int)

# # Calcular estadísticas de completitud
# total_horas = len(todas_horas)
# horas_con_datos = df_completo['tiene_datos'].sum()
# porcentaje_completitud = (horas_con_datos / total_horas) * 100

# print(f"Periodo: {fecha_min} a {fecha_max}")
# print(f"Total de horas en el periodo: {total_horas}")
# print(f"Horas con datos: {horas_con_datos} ({porcentaje_completitud:.2f}%)")
# print(f"Horas sin datos: {total_horas - horas_con_datos} ({100-porcentaje_completitud:.2f}%)")

# # Visualizar huecos temporales por día y hora
# df_completo['fecha'] = df_completo.index.date
# df_completo['hora_dia'] = df_completo.index.hour


# import calplot
# import matplotlib.pyplot as plt

# # Preparar datos para el calendario
# df_completo['fecha'] = pd.to_datetime(df_completo['fecha'])
# datos_diarios = df_completo.groupby('fecha')['tiene_datos'].sum()
# datos_diarios = datos_diarios / 24 * 100  # Convertir a porcentaje de completitud

# # Crear visualización de calendario
# plt.figure(figsize=(16, 10))
# calplot.calplot(datos_diarios, cmap='YlGn', 
#                fillcolor='whitesmoke',
#                vmin=0, vmax=100, 
#                suptitle=f'Disponibilidad diaria de datos (%) - Sensor {10885}')
# plt.tight_layout()
# plt.show()
